# 05. Supervised Classification Modeling & Explainable AI
## Supply Chain Optimization — FMCG / Retail

---

### Objective
1. Benchmark supervised models:
   - **Logistic Regression (Multinomial)**
   - **Decision Tree Classifier**
   - **Random Forest Classifier**
2. Apply **Stratified 5-Fold Cross Validation** on 80% train partition to prevent data leakage.
3. Evaluate on 20% holdout test set: Accuracy, Macro F1, Shortage Recall & Precision.
4. Extract **Odds Ratios** and **MDI Feature Importances** for operational decision-making.


In [ ]:
import sys
sys.path.append("..")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.modeling import InventoryRiskClassifier
from src.evaluation import (
    evaluate_model_performance,
    generate_model_comparison_table,
    extract_logistic_regression_interpretability,
    extract_tree_feature_importances
)

df = pd.read_csv("../data/processed/fmcg_supply_chain_engineered.csv")


### 1. Cross-Validation Benchmarking


In [ ]:
clf_engine = InventoryRiskClassifier(random_state=42)
cv_results_df = clf_engine.train_and_benchmark(df, test_size=0.20, cv_folds=5)
display(cv_results_df)


### 2. Out-of-Sample Holdout Evaluation & Model Selection Matrix


In [ ]:
X_test, y_test = clf_engine.test_data
comparison_df, selected_model = generate_model_comparison_table(
    clf_engine.fitted_pipelines, X_test, y_test
)
display(comparison_df)
print(f"Production Selected Model: {selected_model}")


### 3. Detailed Metrics & Confusion Matrix for Selected Model


In [ ]:
prod_pipeline = clf_engine.fitted_pipelines[selected_model]
eval_metrics = evaluate_model_performance(prod_pipeline, X_test, y_test)

print(f"Test Accuracy: {eval_metrics['accuracy']:.2%}")
print(f"Shortage Recall: {eval_metrics['shortage_recall']:.2%}")
print(f"Shortage Precision: {eval_metrics['shortage_precision']:.2%}")

display(eval_metrics["per_class_metrics"])


In [ ]:
plt.figure(figsize=(6, 5))
sns.heatmap(
    eval_metrics["confusion_matrix"],
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=eval_metrics["classes"],
    yticklabels=eval_metrics["classes"]
)
plt.title(f"Holdout Confusion Matrix: {selected_model}", fontsize=12, fontweight="bold")
plt.xlabel("Predicted Class")
plt.ylabel("Actual Class")
plt.show()


### 4. Model Interpretability: Logistic Regression Odds Ratios


In [ ]:
lr_pipe = clf_engine.fitted_pipelines["Logistic Regression"]
df_odds = extract_logistic_regression_interpretability(lr_pipe)
display(df_odds.head(10))


### 5. Feature Importance: Random Forest Classifier


In [ ]:
rf_pipe = clf_engine.fitted_pipelines["Random Forest"]
df_imp = extract_tree_feature_importances(rf_pipe, top_n=12)

plt.figure(figsize=(10, 5))
sns.barplot(data=df_imp, x="Importance", y="Feature_Clean", palette="viridis")
plt.title("Random Forest MDI Feature Importances", fontsize=12, fontweight="bold")
plt.show()
